In [21]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

# Loading the pre-computed planforms and the fuselage

In [2]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

In [3]:
assumptions = Assumptions()

#TODO load the fuselage here
main_gear_bay = Bay(surface_wetted=np.pi*0.02*0.5, length=0.5, diameter=0.02)
engine_bay = Bay(surface_wetted=np.pi*0.02*0.5, length=0.5, diameter=0.02)
main_gear = LandingGear(wheel_width=assumptions.main_gear_width_wheel, exposed_height=0.02, wheel_diameter=0.03, strut_width=0.01)
nose_gear = LandingGear(wheel_width=assumptions.main_gear_width_wheel, exposed_height=0.02, wheel_diameter=0.03, strut_width=0.01)
fuselage = Fuselage(surface_wetted=np.pi*0.03*3.4, length_total=3.4, diameter_max=0.03, upsweep=np.pi/12, base_area=np.pi/4*0.007**2)
fixed = Fixed(mass=25., fuel_mass=10., x_cg_min=1.5, x_cg_max=1.7, x_tail_cone=2.3, z_cg=0.04, z_tail_cone=0.02, z_wing=0.05, x_LE_canard=0.01,
              x_LE_wing=1.6, x_LE_tail=3.25, x_nose_gear=0.1, x_main_gear=1.8, y_main_gear=0.02, fuselage=fuselage, nose_gear=nose_gear,
              main_gear=main_gear, gear_bay=main_gear_bay, engine_bay=engine_bay)

In [4]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [24]:
aircraft:list[Aircraft] = list()

for planform_recovered in plaforms_recovered:
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, AR_h=max(5., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, AR_c=max(5., main_wing.aspect_ratio/2))

    emp = ef.find_planforms(main_wing)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

In [17]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))

[0.4042341549537764, 0.07761773300654361, 0.4042341549537764, 0.09907364276928177, 0.07354209811878654, 0.02314570143068988, 0.04185995931890717, 0.013492327390536967, 0.4042341549537764, 0.07761773300654361, 0.4042341549537764, 0.09907364276928177, 0.07354209811878654, 0.02314570143068988, 0.04185995931890717, 0.013492327390536967, 0.4042341549537764, 0.07761773300654361, 0.4042341549537764, 0.09907364276928177, 0.07354209811878654, 0.02314570143068988, 0.04185995931890717, 0.013492327390536967, 0.3398835714670884, 0.2954479671847792, 0.3398835714670884, 0.34129325641285657, 0.12102051745707951, 0.09718865782513056, 0.09454650250080399, 0.07844354614285193, 0.3398835714670884, 0.2954479671847792, 0.3398835714670884, 0.34129325641285657, 0.12102051745707951, 0.09718865782513056, 0.09454650250080399, 0.07844354614285193, 0.3398835714670884, 0.2954479671847792, 0.3398835714670884, 0.34129325641285657, 0.12102051745707951, 0.09718865782513056, 0.09454650250080399, 0.07844354614285193, 0.6

# Checking if reuirements are met

In [25]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [26]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 10.0 kg
Fuel required: 7.253120885541449 kg
Difference: 2.746879114458551 kg
theta fails
ac mass: 32.0, 0.7450587549552561
MainWing: AR=5.0, tc=0.06, sweep=-20.0 deg, cmac=-0.05
Failed: ['Landing Gear']

Fuel available: 10.0 kg
Fuel required: 7.1177553982945385 kg
Difference: 2.8822446017054615 kg
theta fails
ac mass: 32.0, 0.7450587549552561
MainWing: AR=5.0, tc=0.06, sweep=-20.0 deg, cmac=-0.05
Failed: ['Landing Gear']

Fuel available: 10.0 kg
Fuel required: 7.253120885541449 kg
Difference: 2.746879114458551 kg
theta fails
ac mass: 32.0, 0.7450587549552561
MainWing: AR=5.0, tc=0.06, sweep=-20.0 deg, cmac=0.05
Failed: ['Landing Gear']

Fuel available: 10.0 kg
Fuel required: 7.087012282196924 kg
Difference: 2.912987717803076 kg
theta fails
ac mass: 32.0, 0.7450587549552561
MainWing: AR=5.0, tc=0.06, sweep=-20.0 deg, cmac=0.05
Failed: ['Landing Gear']

Fuel available: 10.0 kg
Fuel required: 7.5815655709622805 kg
Difference: 2.4184344290377195 kg
theta fails
ac mass: 32.0